# Pharmacokinetic Modeling of Oral Theophylline in Python

**Author:** PATRICIA IJEAMAKA FRANKLIN-UDE  
**Project Type:** PHARMACOKINETIC MODELING AND MODEL COMPARISON 
**Focus:** One-compartment and two-compartment oral PK models using the Theoph dataset

## Overview

This notebook demonstrates compartmental pharmacokinetic modeling using the classic theophiline dataset. 

The project fits a one-compartment oral model and a two-compartment oral model to theophylline concentration-time data.

It evaluates model fit using RMSE, MAE, and R², and compares model performance using AIC and BIC.


## Research Objective

This study aims to model the pharmacokinetics of orally administered theophylline using compartmental approaches and to evaluate whether a **two-compartment model** provides a meaningfully better fit than a **one-compartment model**.

### Research Question
Does a two-compartment oral pharmacokinetic model better describe the concentration-time profile of theophylline than a one-compartment oral model?

### Hypothesis
While the two-compartment model may improve statistical fit, the one-compartment model may remain preferable when model simplicity and interpretability are considered.

## Clinical Context

Theophylline is a bronchodilator used in respiratory disease management. Because it significantly varies clinically in exposure, pharmacokinetic modeling helps quantify:

- absorption
- elimination
- peak concentration
- drug exposure over time

Understanding these features is important for dose optimization and safe therapy.

## Dataset Description

The analysis uses the Theophiline dataset, a standard pharmacokinetic dataset from R.

### Dataset facts
- **132 observations**
- **12 subjects**
- **11 time points per subject**
- sampling over **25 hours** after oral dosing

### Variables
- **Subject**: subject identifier
- **Wt**: body weight (kg)
- **Dose**: oral dose (mg/kg)
- **Time**: time after dose (hours)
- **conc**: measured concentration (mg/L)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

pd.set_option("display.float_format", lambda x: f"{x:.4f}")

ModuleNotFoundError: No module named 'numpy'

In [ ]:
# Load dataset
url = "https://vincentarelbundock.github.io/Rdatasets/csv/datasets/Theoph.csv"
df = pd.read_csv(url)

if "rownames" in df.columns:
    df = df.drop(columns=["rownames"])

df["Subject"] = df["Subject"].astype(int)
df = df.dropna().copy()

print("Dataset shape:", df.shape)
print("Number of subjects:", df["Subject"].nunique())
df.head()

In [ ]:
print("Columns:", df.columns.tolist())
print("Time range:", df["Time"].min(), "to", df["Time"].max(), "hours")
print("Concentration range:", df["conc"].min(), "to", df["conc"].max(), "mg/L")
print("Dose values (mg/kg):", sorted(df["Dose"].unique()))

## Exploratory Analysis

Before fitting models, concentration-time profiles are visualized to inspect:

- absorption phase
- peak concentration
- elimination phase
- between-subject variability

In [ ]:
plt.figure(figsize=(9, 6))

for sid, sub in df.groupby("Subject"):
    plt.plot(sub["Time"], sub["conc"], marker="o", linewidth=1, label=f"Subject {sid}")

plt.xlabel("Time (h)")
plt.ylabel("Concentration (mg/L)")
plt.title("Theophylline Concentration-Time Profiles by Subject")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("figures/concentration_time_profiles.png", dpi=300, bbox_inches="tight")
plt.show()

### Exploratory Interpretation

The concentration-time profiles show clear absorption and elimination phases, with visible variability across subjects. This supports the use of compartmental modeling and motivates comparison of simple and more flexible PK structures.

## One-Compartment Model

The one-compartment oral model assumes:
- first-order absorption
- first-order elimination
- the body behaves as a single well-mixed compartment

### Model equation

\[
C(t) = \frac{Dose \cdot k_a}{(V/F)(k_a-k_e)} \left(e^{-k_e t} - e^{-k_a t}\right)
\]

### Parameters
- \(k_a\): absorption rate constant
- \(k_e\): elimination rate constant
- \(V/F\): apparent volume of distribution

In [ ]:
def oral_one_compartment(t, ka, ke, V_over_F, dose_mg):
    if np.isclose(ka, ke):
        ka = ke + 1e-6
    return (dose_mg * ka / (V_over_F * (ka - ke))) * (np.exp(-ke * t) - np.exp(-ka * t))

## Parameter Estimation

The one-compartment model is first fitted to a single subject to estimate:
- absorption rate constant
- elimination rate constant
- apparent volume of distribution

This provides a clear subject-level demonstration before scaling to all subjects.

In [ ]:
subject_id = 1
sub = df[df["Subject"] == subject_id].sort_values("Time").copy()

time = sub["Time"].values
conc = sub["conc"].values
wt = sub["Wt"].iloc[0]
dose_mg_per_kg = sub["Dose"].iloc[0]
dose_mg = wt * dose_mg_per_kg

print(f"Subject: {subject_id}")
print(f"Weight: {wt:.2f} kg")
print(f"Dose: {dose_mg_per_kg:.2f} mg/kg")
print(f"Total dose: {dose_mg:.2f} mg")

sub

In [ ]:
p0_one = [1.0, 0.1, 30.0]
bounds_one = ([0.01, 0.001, 1.0], [5.0, 2.0, 500.0])

params_one, cov_one = curve_fit(
    lambda t, ka, ke, V_over_F: oral_one_compartment(t, ka, ke, V_over_F, dose_mg),
    time,
    conc,
    p0=p0_one,
    bounds=bounds_one,
    maxfev=20000
)

ka1, ke1, v1 = params_one

print("Estimated one-compartment parameters")
print(f"ka   = {ka1:.4f} 1/h")
print(f"ke   = {ke1:.4f} 1/h")
print(f"V/F  = {v1:.4f} L")

In [ ]:
t_grid = np.linspace(0, 25, 400)
pred_one_dense = oral_one_compartment(t_grid, ka1, ke1, v1, dose_mg)

half_life_1 = np.log(2) / ke1
cmax_1 = pred_one_dense.max()
tmax_1 = t_grid[np.argmax(pred_one_dense)]
auc_0_25_1 = np.trapz(pred_one_dense, t_grid)

pk_metrics_one = pd.DataFrame({
    "Metric": ["ka (1/h)", "ke (1/h)", "V/F (L)", "Half-life (h)", "Cmax (mg/L)", "Tmax (h)", "AUC(0-25) (mg·h/L)"],
    "Value": [ka1, ke1, v1, half_life_1, cmax_1, tmax_1, auc_0_25_1]
})

pk_metrics_one

### Pharmacokinetic Interpretation

- A larger **ka** suggests faster absorption.
- A larger **ke** suggests faster elimination.
- **V/F** reflects the extent of apparent drug distribution.

These parameter estimates summarize subject-level theophylline disposition and provide the basis for formal goodness-of-fit evaluation.

## Goodness-of-Fit

Model fit is evaluated using:
- **RMSE**: Root Mean Squared Error
- **MAE**: Mean Absolute Error
- **R²**: coefficient of determination

These metrics quantify prediction error and agreement between observed and predicted concentrations.

In [ ]:
def calculate_gof(observed, predicted, n_params):
    n = len(observed)
    residuals = observed - predicted
    sse = np.sum(residuals ** 2)

    rmse = np.sqrt(mean_squared_error(observed, predicted))
    mae = mean_absolute_error(observed, predicted)
    r2 = r2_score(observed, predicted)

    if sse <= 0:
        sse = 1e-12

    aic = n * np.log(sse / n) + 2 * n_params
    bic = n * np.log(sse / n) + n_params * np.log(n)

    return {
        "SSE": sse,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "AIC": aic,
        "BIC": bic
    }

In [ ]:
pred_one_obs = oral_one_compartment(time, ka1, ke1, v1, dose_mg)
gof_one = calculate_gof(conc, pred_one_obs, n_params=3)

gof_one_df = pd.DataFrame({
    "Metric": list(gof_one.keys()),
    "Value": list(gof_one.values())
})

gof_one_df

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(conc, pred_one_obs)
plt.plot([conc.min(), conc.max()], [conc.min(), conc.max()], linestyle="--")
plt.xlabel("Observed Concentration (mg/L)")
plt.ylabel("Predicted Concentration (mg/L)")
plt.title(f"Observed vs Predicted — One-Compartment (Subject {subject_id})")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("figures/observed_vs_predicted_subject1.png", dpi=300, bbox_inches="tight")
plt.show()

## Residual Analysis

Residual analysis helps assess whether the model captures the data without systematic bias.

A good fit typically shows:
- residuals centered around zero
- no clear time-dependent pattern

In [ ]:
residuals_one = conc - pred_one_obs
normalized_residuals_one = residuals_one / np.std(residuals_one)

residual_df_one = pd.DataFrame({
    "Time (h)": time,
    "Observed": conc,
    "Predicted": pred_one_obs,
    "Residual": residuals_one,
    "Normalized Residual": normalized_residuals_one
})

residual_df_one

In [ ]:
plt.figure(figsize=(9, 5))
plt.scatter(time, residuals_one)
plt.axhline(0, linestyle="--")
plt.xlabel("Time (h)")
plt.ylabel("Residual (Observed - Predicted)")
plt.title(f"Residual Plot — One-Compartment (Subject {subject_id})")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("figures/residual_plot_subject1.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
plt.scatter(time, normalized_residuals_one)
plt.axhline(0, linestyle="--")
plt.xlabel("Time (h)")
plt.ylabel("Normalized Residual")
plt.title(f"Normalized Residual Plot — One-Compartment (Subject {subject_id})")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("figures/normalized_residual_plot_subject1.png", dpi=300, bbox_inches="tight")
plt.show()

### Residual Interpretation

A random spread of residuals around zero supports model adequacy. Any systematic clustering across time can suggest structural mismatch and motivate a more complex model.

## Two-Compartment Model

The two-compartment oral model allows drug distribution between:
- a central compartment
- a peripheral compartment

This model can better capture early distribution behavior but introduces more parameters.

In [ ]:
def oral_two_compartment(t, ka, alpha, beta, A, B):
    return A * (np.exp(-alpha * t) - np.exp(-ka * t)) + \
           B * (np.exp(-beta * t) - np.exp(-ka * t))

In [ ]:
p0_two = [1.0, 1.0, 0.1, 8.0, 4.0]
bounds_two = (
    [0.01, 0.01, 0.001, 0.001, 0.001],
    [5.0, 5.0, 2.0, 100.0, 100.0]
)

params_two, cov_two = curve_fit(
    oral_two_compartment,
    time,
    conc,
    p0=p0_two,
    bounds=bounds_two,
    maxfev=50000
)

ka2, alpha2, beta2, A2, B2 = params_two
pred_two_obs = oral_two_compartment(time, ka2, alpha2, beta2, A2, B2)
pred_two_dense = oral_two_compartment(t_grid, ka2, alpha2, beta2, A2, B2)

print("Estimated two-compartment parameters")
print(f"ka    = {ka2:.4f}")
print(f"alpha = {alpha2:.4f}")
print(f"beta  = {beta2:.4f}")
print(f"A     = {A2:.4f}")
print(f"B     = {B2:.4f}")

In [ ]:
gof_two = calculate_gof(conc, pred_two_obs, n_params=5)

gof_two_df = pd.DataFrame({
    "Metric": list(gof_two.keys()),
    "Value": list(gof_two.values())
})

gof_two_df

## Model Comparison

The one-compartment and two-compartment models are compared using:
- RMSE
- MAE
- R²
- AIC
- BIC

Lower RMSE, MAE, AIC, and BIC indicate better fit, while higher R² indicates stronger agreement between observed and predicted concentrations.

In [ ]:
comparison_df = pd.DataFrame([
    {"Model": "One-compartment", **gof_one},
    {"Model": "Two-compartment", **gof_two}
])

comparison_df

In [ ]:
plt.figure(figsize=(9, 6))
plt.scatter(time, conc, label="Observed data")
plt.plot(t_grid, pred_one_dense, label="One-compartment fit")
plt.plot(t_grid, pred_two_dense, label="Two-compartment fit")
plt.xlabel("Time (h)")
plt.ylabel("Concentration (mg/L)")
plt.title(f"Model Comparison — Subject {subject_id}")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig("figures/model_comparison_subject1.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(conc, pred_one_obs)
axes[0].plot([conc.min(), conc.max()], [conc.min(), conc.max()], linestyle="--")
axes[0].set_title("One-Compartment")
axes[0].set_xlabel("Observed")
axes[0].set_ylabel("Predicted")
axes[0].grid(True, alpha=0.3)

axes[1].scatter(conc, pred_two_obs)
axes[1].plot([conc.min(), conc.max()], [conc.min(), conc.max()], linestyle="--")
axes[1].set_title("Two-Compartment")
axes[1].set_xlabel("Observed")
axes[1].set_ylabel("Predicted")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("figures/one_vs_two_compartment_fit.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
residuals_two = conc - pred_two_obs

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(time, residuals_one)
axes[0].axhline(0, linestyle="--")
axes[0].set_title("Residuals — One-Compartment")
axes[0].set_xlabel("Time (h)")
axes[0].set_ylabel("Residual")
axes[0].grid(True, alpha=0.3)

axes[1].scatter(time, residuals_two)
axes[1].axhline(0, linestyle="--")
axes[1].set_title("Residuals — Two-Compartment")
axes[1].set_xlabel("Time (h)")
axes[1].set_ylabel("Residual")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("figures/residuals_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

### Model Comparison Interpretation

The two-compartment model may reduce error by capturing distribution behaviour more flexibly. However, model selection should also consider complexity. AIC and BIC help determine whether the added parameters are justified.
These findings emphasise that increased model complexity does not necessarily guarantee better generalisation, particularly when evaluated using penalised model selection criteria.

In [ ]:
results_one = []

for sid, g in df.groupby("Subject"):
    g = g.sort_values("Time")
    time_i = g["Time"].values
    conc_i = g["conc"].values
    wt_i = g["Wt"].iloc[0]
    dose_i = g["Dose"].iloc[0] * wt_i

    try:
        p_one, _ = curve_fit(
            lambda t, ka, ke, V_over_F: oral_one_compartment(t, ka, ke, V_over_F, dose_i),
            time_i,
            conc_i,
            p0=[1.0, 0.1, 30.0],
            bounds=([0.01, 0.001, 1.0], [5.0, 2.0, 500.0]),
            maxfev=20000
        )

        ka_i, ke_i, v_i = p_one
        pred_obs_i = oral_one_compartment(time_i, ka_i, ke_i, v_i, dose_i)
        pred_dense_i = oral_one_compartment(t_grid, ka_i, ke_i, v_i, dose_i)
        gof_i = calculate_gof(conc_i, pred_obs_i, n_params=3)

        results_one.append({
            "Subject": sid,
            "Wt_kg": wt_i,
            "Dose_mg": dose_i,
            "ka_1_per_h": ka_i,
            "ke_1_per_h": ke_i,
            "V_over_F_L": v_i,
            "half_life_h": np.log(2) / ke_i,
            "Cmax_mg_per_L": pred_dense_i.max(),
            "Tmax_h": t_grid[np.argmax(pred_dense_i)],
            "AUC_0_25_mg_h_per_L": np.trapz(pred_dense_i, t_grid),
            "RMSE": gof_i["RMSE"],
            "MAE": gof_i["MAE"],
            "R2": gof_i["R2"],
            "AIC": gof_i["AIC"],
            "BIC": gof_i["BIC"]
        })
    except Exception as e:
        print(f"Subject {sid} failed for one-compartment fit: {e}")

results_one_df = pd.DataFrame(results_one)
results_one_df

In [ ]:
model_compare_results = []

for sid, g in df.groupby("Subject"):
    g = g.sort_values("Time")
    time_i = g["Time"].values
    conc_i = g["conc"].values
    wt_i = g["Wt"].iloc[0]
    dose_i = g["Dose"].iloc[0] * wt_i

    row = {"Subject": sid}

    try:
        p_one, _ = curve_fit(
            lambda t, ka, ke, V_over_F: oral_one_compartment(t, ka, ke, V_over_F, dose_i),
            time_i,
            conc_i,
            p0=[1.0, 0.1, 30.0],
            bounds=([0.01, 0.001, 1.0], [5.0, 2.0, 500.0]),
            maxfev=20000
        )
        pred_one_i = oral_one_compartment(time_i, *p_one, dose_i)
        gof1 = calculate_gof(conc_i, pred_one_i, n_params=3)

        row.update({
            "One_RMSE": gof1["RMSE"],
            "One_MAE": gof1["MAE"],
            "One_R2": gof1["R2"],
            "One_AIC": gof1["AIC"],
            "One_BIC": gof1["BIC"]
        })
    except Exception:
        row.update({
            "One_RMSE": np.nan,
            "One_MAE": np.nan,
            "One_R2": np.nan,
            "One_AIC": np.nan,
            "One_BIC": np.nan
        })

    try:
        p_two, _ = curve_fit(
            oral_two_compartment,
            time_i,
            conc_i,
            p0=[1.0, 1.0, 0.1, 8.0, 4.0],
            bounds=([0.01, 0.01, 0.001, 0.001, 0.001], [5.0, 5.0, 2.0, 100.0, 100.0]),
            maxfev=50000
        )
        pred_two_i = oral_two_compartment(time_i, *p_two)
        gof2 = calculate_gof(conc_i, pred_two_i, n_params=5)

        row.update({
            "Two_RMSE": gof2["RMSE"],
            "Two_MAE": gof2["MAE"],
            "Two_R2": gof2["R2"],
            "Two_AIC": gof2["AIC"],
            "Two_BIC": gof2["BIC"]
        })
    except Exception:
        row.update({
            "Two_RMSE": np.nan,
            "Two_MAE": np.nan,
            "Two_R2": np.nan,
            "Two_AIC": np.nan,
            "Two_BIC": np.nan
        })

    model_compare_results.append(row)

model_compare_df = pd.DataFrame(model_compare_results)
model_compare_df

In [ ]:
def choose_better_model(row):
    if pd.isna(row["One_AIC"]) or pd.isna(row["Two_AIC"]):
        return "Undetermined"
    return "Two-compartment" if row["Two_AIC"] < row["One_AIC"] else "One-compartment"

model_compare_df["Better_Model_By_AIC"] = model_compare_df.apply(choose_better_model, axis=1)
model_compare_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(results_one_df["Subject"].astype(str), results_one_df["RMSE"])
plt.xlabel("Subject")
plt.ylabel("RMSE")
plt.title("One-Compartment RMSE by Subject")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("figures/rmse_by_subject.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(results_one_df["Subject"].astype(str), results_one_df["R2"])
plt.xlabel("Subject")
plt.ylabel("R²")
plt.title("One-Compartment R² by Subject")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("figures/r2_by_subject.png", dpi=300, bbox_inches="tight")
plt.show()

## Inter-Individual Variability

Pharmacokinetic parameters often vary between subjects due to differences in:
- body weight
- metabolism
- clearance
- tissue distribution

Evaluating subject-level parameter estimates helps reveal this variability.

In [ ]:
subject_weights = df.groupby("Subject")["Wt"].first().reset_index()
merged_variability = results_one_df.merge(subject_weights, on="Subject")

plt.figure(figsize=(8, 5))
plt.scatter(merged_variability["Wt"], merged_variability["V_over_F_L"])
plt.xlabel("Body Weight (kg)")
plt.ylabel("Estimated V/F (L)")
plt.title("Body Weight vs Apparent Volume of Distribution")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("figures/weight_vs_vf.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
merged_variability[["Subject", "Wt", "ka_1_per_h", "ke_1_per_h", "V_over_F_L", "half_life_h", "Cmax_mg_per_L"]]

### Variability Interpretation

Observed differences in half-life, Cmax, and distribution volume suggest meaningful between-subject variability. This supports the broader pharmacology principle that individuals can differ substantially in drug exposure and response.

## Limitations

This notebook has several important limitations:

- The dataset is small (**12 subjects**).
- The models are simplified compartmental approximations.
- No covariates beyond body weight are modeled.
- The two-compartment parameterization used here is practical for comparison, but not a full population PK analysis.
- Parameter estimates can be sensitive to initial values and optimization settings.

## Future Work

Potential extensions include:

- population pharmacokinetic modeling
- covariate analysis
- Bayesian parameter estimation
- machine learning prediction of PK parameters
- comparison with nonlinear mixed-effects approaches

## Conclusion

## Conclusion

This study implemented and compared one-compartment and two-compartment pharmacokinetic models for oral theophylline data using Python-based computational methods.

The results show that while the two-compartment model can improve fit in certain cases, the one-compartment model often provides a more parsimonious representation when evaluated using AIC and BIC.

This highlights the importance of balancing model complexity with interpretability in pharmacokinetic modelling.

Overall, this project demonstrates the integration of pharmacokinetic principles, nonlinear regression, and data analysis techniques and provides a foundation for further work in pharmacometrics and data-driven drug modelling.


In [ ]:
results_one_df.to_csv("pk_subject_summary_with_gof.csv", index=False)
model_compare_df.to_csv("pk_model_comparison_all_subjects.csv", index=False)
pk_metrics_one.to_csv("pk_subject1_metrics.csv", index=False)
residual_df_one.to_csv("pk_subject1_residuals.csv", index=False)

print("Saved:")
print("- pk_subject_summary_with_gof.csv")
print("- pk_model_comparison_all_subjects.csv")
print("- pk_subject1_metrics.csv")
print("- pk_subject1_residuals.csv")

## Project Summary

This project demonstrates the application of pharmacokinetic modelling concepts using real-world data and highlights the value of computational tools in understanding drug absorption and elimination.

The workflow integrates domain knowledge from pharmacy with data science techniques, making it relevant for further research in pharmacometrics, drug development, and quantitative pharmacology.